In [ ]:
# -*- coding: utf-8 -*-
"""Area > 10000 m²: 30–300 m 随机森林嵌套空间交叉验证及最终调优。

直接在 Jupyter 中运行本文件（%run "完整路径.py"）或复制全文到单元格。
不使用 argparse，不会出现 ipykernel 的 -f 参数错误。

方法与原脚本一致：1000 m 方形网格、外层 GroupKFold 评价、内层
GroupKFold 调参。增加：20 个目标×尺度组合、共同完整样本、逐折保存、
断点续跑、全样本最终调参和模型导出。坐标必须为 EPSG:32650，单位米。

注意：空间块不同不代表训练/测试点之间存在最小隔离距离；本脚本没有
额外的空间排除带，亦不保证完全消除空间相关性。不要把 30–300 m 的
景观缓冲距离与 1000 m 的空间验证块大小混为一谈。

依赖：numpy pandas scipy scikit-learn xlrd openpyxl joblib。
"""

from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import hashlib
import json
import platform
import time

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, RandomizedSearchCV


# =============================================================================
# 1. 用户设置（默认正式运行；不需要修改后面的函数）
# =============================================================================
@dataclass
class Config:
    data_path: str = (
        r"E:\博士申请\第二篇论文\data\wuhuanshiliang\SH_attributes_10000.xls"
    )
    # 独立新文件夹：不覆盖此前的随机划分验证/SHAP结果。
    output_dir: str = (
        r"E:\博士申请\第二篇论文\脚本\Python\importance"
        r"\multiscale_spatial_cv_area_gt10000_ISF"
    )
    sheet_name: int = 0
    area_column: str = "Shape_Area"  # 实际面积字段，不是字面上的 AREA
    minimum_area: float = 10000.0   # 严格大于；恰好10000也不纳入
    targets: tuple = ("SUNNY", "HEATWAVE")
    distances: tuple = tuple(range(30, 301, 30))
    coordinate_columns: tuple = ("CENT_X", "CENT_Y")
    fixed_vars: tuple = (
        "TCC", "GCI", "BCI", "WCI", "RCI", "Shape_Area",
        "jungong", "BAH", "BHSD", "DIST",
    )
    buffer_prefixes: tuple = ("TCC", "GCI", "BCI", "WCI", "ISF", "BAH", "BHSD")
    random_state: int = 123
    block_size: float = 1000.0
    outer_folds: int = 5
    inner_folds: int = 5
    n_iter_search: int = 50
    # 保留原脚本的参数搜索范围；这里上下界均包含。
    n_estimators_min: int = 500
    n_estimators_max: int = 2000
    max_features_min: int = 3
    min_samples_leaf_min: int = 1
    min_samples_leaf_max: int = 15
    # 只在搜索层并行；森林内部单线程，避免两层同时 n_jobs=-1。
    search_jobs: int = 4         # 内存不足时改为2或1；-1为全部CPU
    search_verbose: int = 1
    save_final_models: bool = True
    export_excel: bool = True
    resume: bool = True         # 相同数据和设置时，跳过已完成的折/最终调优
    dry_run: bool = False       # True只检查数据、空间折并输出审计文件，不训练


CONFIG = Config()
SCRIPT_VERSION = "1.0.0"


# =============================================================================
# 2. 通用保存、读入、审计
# =============================================================================
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value)}")


def save_json(value, path):
    """先写临时文件再替换：只把完整结果视为断点。"""
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                               default=json_default), encoding="utf-8")
    temp.replace(path)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def save_csv(frame, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8-sig")
    temp.replace(path)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def features_for_distance(distance, cfg):
    # 同一模型只纳入该距离的7个外部变量，不同时纳入其他距离。
    return list(cfg.fixed_vars) + [f"{v}_b{distance}m" for v in cfg.buffer_prefixes]


def prepare_data(cfg):
    path = Path(cfg.data_path)
    if not path.is_file():
        raise FileNotFoundError(f"找不到数据文件，请修改 CONFIG.data_path：{path}")
    if cfg.block_size <= 0 or min(cfg.outer_folds, cfg.inner_folds) < 2:
        raise ValueError("空间块边长须>0，内外层折数须>=2。")
    if not cfg.distances or len(set(cfg.distances)) != len(cfg.distances):
        raise ValueError("distances不能为空或包含重复距离。")
    if not cfg.targets or len(set(cfg.targets)) != len(cfg.targets):
        raise ValueError("targets不能为空或重复。")
    engine = "xlrd" if path.suffix.lower() == ".xls" else "openpyxl"
    raw = pd.read_excel(path, sheet_name=cfg.sheet_name, engine=engine)
    required = list(dict.fromkeys(
        [cfg.area_column] + list(cfg.targets) + list(cfg.coordinate_columns)
        + [v for d in cfg.distances for v in features_for_distance(d, cfg)]
    ))
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise KeyError("缺少必需字段：\n" + "\n".join(missing))
    numeric = raw[required].apply(pd.to_numeric, errors="coerce")
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    for target in cfg.targets:
        numeric.loc[numeric[target] == 0, target] = np.nan  # 延续原脚本约定

    area_ok = numeric[cfg.area_column] > cfg.minimum_area
    complete = numeric.notna().all(axis=1)
    included = area_ok & complete
    # 用Excel行号作唯一追踪键；不依据可能重复的社区名称拼接结果。
    audit = pd.DataFrame({"source_excel_row": np.arange(len(raw)) + 2})
    for name in ("name", "NAME", "FID", "OBJECTID"):
        if name in raw.columns:
            audit[name] = raw[name].to_numpy()
    audit[cfg.area_column] = numeric[cfg.area_column].to_numpy()
    audit["area_gt_minimum"] = area_ok.to_numpy()
    audit["complete_all_targets_all_scales"] = complete.to_numpy()
    audit["included"] = included.to_numpy()
    audit["invalid_or_missing_columns"] = numeric.isna().apply(
        lambda row: ";".join(row.index[row]), axis=1
    )
    audit["exclusion_reason"] = np.select(
        [numeric[cfg.area_column].isna(), ~area_ok, ~complete],
        ["missing_or_invalid_area", "area_not_greater_than_minimum",
         "incomplete_required_columns"], default="included"
    )
    data = numeric.loc[included].reset_index(drop=True)
    identities = audit.loc[included].reset_index(drop=True)
    if len(data) < 50:
        raise ValueError(f"共同完整案例只有{len(data)}个，少于50。")
    for target in cfg.targets:
        if data[target].nunique() < 2:
            raise ValueError(f"{target}无变异，无法评价R²。")
    xy = data[list(cfg.coordinate_columns)].to_numpy(dtype=float)
    if np.max(np.abs(xy[:, 0])) <= 180 and np.max(np.abs(xy[:, 1])) <= 90:
        raise ValueError("坐标看起来是经纬度；须先转换到以米为单位的EPSG:32650。")
    origin = xy.min(axis=0)
    block_xy = np.floor((xy - origin) / cfg.block_size).astype(np.int64)
    labels = pd.Series(list(map(tuple, block_xy)))
    groups = pd.factorize(labels, sort=True)[0]
    if len(np.unique(groups)) < cfg.outer_folds:
        raise ValueError("有效空间组数量少于外层折数。")
    outer_splits = list(GroupKFold(n_splits=cfg.outer_folds).split(data, groups=groups))
    fold_assignment = np.zeros(len(data), dtype=int)
    fold_plan = []
    for fold, (train, test) in enumerate(outer_splits, 1):
        assert not np.intersect1d(groups[train], groups[test]).size
        n_inner = min(cfg.inner_folds, len(np.unique(groups[train])))
        if n_inner < 2 or len(test) < 2:
            raise ValueError("空间组/样本不足，不能执行指定的嵌套验证。")
        fold_assignment[test] = fold
        fold_plan.append({"outer_fold": fold, "n_train": len(train),
                          "n_test": len(test), "n_train_blocks": len(np.unique(groups[train])),
                          "n_test_blocks": len(np.unique(groups[test])), "inner_folds": n_inner})
    identities["CENT_X"] = xy[:, 0]
    identities["CENT_Y"] = xy[:, 1]
    identities["block_x"] = block_xy[:, 0]
    identities["block_y"] = block_xy[:, 1]
    identities["spatial_group"] = groups
    identities["outer_fold"] = fold_assignment
    report = {
        "n_input": len(raw), "n_area_eligible": int(area_ok.sum()),
        "n_common_complete": len(data), "n_spatial_blocks": len(np.unique(groups)),
        "n_excluded_area_or_missing_area": int((~area_ok).sum()),
        "n_excluded_incomplete_after_area_filter": int((area_ok & ~complete).sum()),
        "area_column": cfg.area_column, "minimum_area_exclusive_m2": cfg.minimum_area,
        "grid_origin_x": float(origin[0]), "grid_origin_y": float(origin[1]),
        "block_size_m": cfg.block_size,
        "common_sample_rule": "Complete for both targets, coordinates and every selected scale",
    }
    return data, identities, audit, groups, outer_splits, pd.DataFrame(fold_plan), report


def initialize_output(cfg, report):
    root = Path(cfg.output_dir)
    root.mkdir(parents=True, exist_ok=True)
    settings = asdict(cfg)
    # 仅这些运行控制项改变时，可以继续复用同一计算结果。
    for key in ("output_dir", "resume", "dry_run", "search_jobs", "search_verbose", "export_excel"):
        settings.pop(key)
    versions = {"python": platform.python_version(), "numpy": np.__version__,
                "pandas": pd.__version__, "scipy": scipy.__version__,
                "sklearn": sklearn.__version__, "joblib": joblib.__version__}
    signature_inputs = {"script_version": SCRIPT_VERSION, "settings": settings,
                        "input_sha256": file_sha256(cfg.data_path), "versions": versions}
    signature = hashlib.sha256(json.dumps(signature_inputs, sort_keys=True,
                                          default=json_default).encode("utf-8")).hexdigest()
    manifest_path = root / "run_metadata.json"
    if manifest_path.exists():
        old = load_json(manifest_path)
        if old["signature"] != signature:
            raise ValueError("输出目录存在不同数据/参数/版本的结果。请更换 output_dir，避免混用。")
        if not cfg.resume:
            raise ValueError("目录已有本次结果；使用resume=True续跑，或换新输出目录重跑。")
    else:
        if any(root.iterdir()):
            raise ValueError("输出目录非空且没有运行记录。请为本脚本选择一个新的子文件夹。")
        save_json({"signature": signature, **signature_inputs, "sample_report": report,
                   "created_utc": datetime.now(timezone.utc).isoformat()}, manifest_path)
    return root


# =============================================================================
# 3. 空间调参：所有尺度使用相同的候选参数抽样、随机种子和数据划分
# =============================================================================
def build_search(n_features, n_splits, cfg):
    if cfg.n_iter_search < 1 or cfg.search_jobs == 0:
        raise ValueError("n_iter_search须>=1，search_jobs不能为0。")
    for low, high in ((cfg.n_estimators_min, cfg.n_estimators_max),
                      (cfg.min_samples_leaf_min, cfg.min_samples_leaf_max),
                      (cfg.max_features_min, n_features)):
        if low < 1 or low > high:
            raise ValueError("参数搜索上下界无效。")
    return RandomizedSearchCV(
        estimator=RandomForestRegressor(bootstrap=True, random_state=cfg.random_state, n_jobs=1),
        param_distributions={
            "n_estimators": randint(cfg.n_estimators_min, cfg.n_estimators_max + 1),
            "max_features": randint(cfg.max_features_min, n_features + 1),
            "min_samples_leaf": randint(cfg.min_samples_leaf_min, cfg.min_samples_leaf_max + 1),
        },
        n_iter=cfg.n_iter_search, scoring="neg_root_mean_squared_error",
        cv=GroupKFold(n_splits=n_splits), random_state=cfg.random_state,
        n_jobs=cfg.search_jobs, pre_dispatch="n_jobs", verbose=cfg.search_verbose,
        refit=True, error_score="raise", return_train_score=False,
    )


def metrics(observed, predicted):
    return {
        "R2": float(r2_score(observed, predicted)) if len(observed) > 1 and np.var(observed) > 0 else None,
        "RMSE": float(np.sqrt(mean_squared_error(observed, predicted))),
    }


def save_search_results(search, path):
    results = pd.DataFrame(search.cv_results_)
    results["mean_validation_RMSE"] = -results["mean_test_score"]
    results["std_validation_RMSE"] = results["std_test_score"]
    save_csv(results, path)


def run_combination(data, identities, groups, outer_splits, target, distance, root, cfg):
    folder = root / f"buffer_{distance:03d}m" / target
    folder.mkdir(parents=True, exist_ok=True)
    features = features_for_distance(distance, cfg)
    X = data[features]
    y = data[target].to_numpy(dtype=float)
    predictions = np.full(len(y), np.nan)
    folds = []
    for fold, (train, test) in enumerate(outer_splits, 1):
        fold_dir = folder / f"outer_fold_{fold:02d}"
        fold_dir.mkdir(exist_ok=True)
        checkpoint = fold_dir / "completed.json"
        if checkpoint.exists() and cfg.resume:
            record = load_json(checkpoint)
            if not np.array_equal(record["test_indices"], test):
                raise RuntimeError("断点测试索引与当前空间折不一致。")
            print(f"[{target}, {distance}m] 外层折{fold}：读取已完成断点", flush=True)
        else:
            n_inner = min(cfg.inner_folds, len(np.unique(groups[train])))
            print(f"[{target}, {distance}m] 开始外层折{fold}/{len(outer_splits)}，"
                  f"内层{n_inner}折×{cfg.n_iter_search}组参数", flush=True)
            started = time.monotonic()
            search = build_search(len(features), n_inner, cfg)
            search.fit(X.iloc[train], y[train], groups=groups[train])
            predicted = search.best_estimator_.predict(X.iloc[test])
            scores = metrics(y[test], predicted)
            record = {
                "LST": target, "Buffer_m": distance, "outer_fold": fold,
                "n_train": len(train), "n_test": len(test),
                "n_train_blocks": len(np.unique(groups[train])),
                "n_test_blocks": len(np.unique(groups[test])), "inner_folds": n_inner,
                "test_R2": scores["R2"], "test_RMSE": scores["RMSE"],
                "inner_best_CV_RMSE": float(-search.best_score_),
                "best_params": search.best_params_,
                "elapsed_seconds": time.monotonic() - started,
                "test_indices": test.tolist(), "predictions": predicted.tolist(),
            }
            save_search_results(search, fold_dir / "inner_search_results.csv")
            # 最后写completed.json：如果中途退出，只重跑尚未完成的一折。
            save_json(record, checkpoint)
            print(f"  完成：外层RMSE={scores['RMSE']:.4f}，"
                  f"用时{record['elapsed_seconds']/60:.1f}分钟", flush=True)
        predicted = np.asarray(record["predictions"], dtype=float)
        if len(predicted) != len(test) or not np.isfinite(predicted).all():
            raise RuntimeError(f"断点预测无效：{checkpoint}")
        predictions[test] = predicted
        folds.append({k: v for k, v in record.items()
                      if k not in ("test_indices", "predictions", "best_params")} | record["best_params"])
    if not np.isfinite(predictions).all():
        raise RuntimeError("存在未获得外层OOF预测的社区。")
    pooled = metrics(y, predictions)
    prediction_table = identities.copy()
    prediction_table["LST"] = target
    prediction_table["Buffer_m"] = distance
    prediction_table["observed_LST"] = y
    prediction_table["predicted_LST_OOF"] = predictions
    prediction_table["residual_observed_minus_predicted"] = y - predictions
    save_csv(prediction_table, folder / "outer_oof_predictions.csv")
    save_csv(pd.DataFrame(folds), folder / "outer_fold_metrics_and_params.csv")
    summary = {
        "LST": target, "Buffer_m": distance, "n_samples": len(y),
        "n_spatial_blocks": len(np.unique(groups)),
        "Spatial_CV_R2": pooled["R2"], "Spatial_CV_RMSE": pooled["RMSE"],
        "outer_folds": len(outer_splits), "Block_size_m": cfg.block_size,
    }
    save_json(summary, folder / "nested_spatial_cv_metrics.json")

    # 独立阶段：在全部共同样本中进行空间CV调参、全样本refit，供后续SHAP/情景模拟使用。
    # 不能用此阶段的调参最优分数替代外层OOF性能。
    final_checkpoint = folder / "final_tuning_completed.json"
    model_path = folder / "final_rf_model.joblib"
    if final_checkpoint.exists() and cfg.resume:
        final = load_json(final_checkpoint)
        if cfg.save_final_models and not model_path.is_file():
            raise FileNotFoundError(f"最终模型丢失：{model_path}；请换新输出目录重新运行。")
        print(f"[{target}, {distance}m] 最终调优：读取已完成断点", flush=True)
    else:
        n_final_folds = min(cfg.inner_folds, len(np.unique(groups)))
        print(f"[{target}, {distance}m] 开始全样本空间调优（用于最终参数，不用于独立评价）", flush=True)
        started = time.monotonic()
        final_search = build_search(len(features), n_final_folds, cfg)
        final_search.fit(X, y, groups=groups)
        final_model = final_search.best_estimator_
        save_search_results(final_search, folder / "final_search_results.csv")
        final = {
            "LST": target, "Buffer_m": distance, "n_samples": len(y),
            "features_in_order": features, "best_params": final_search.best_params_,
            "all_rf_params": final_model.get_params(),
            "final_tuning_best_CV_RMSE": float(-final_search.best_score_),
            "final_tuning_folds": n_final_folds, "elapsed_seconds": time.monotonic() - started,
            "score_note": "Selection score, NOT independent performance; use nested OOF metrics.",
        }
        save_json(final, folder / "final_best_params.json")
        if cfg.save_final_models:
            temp_model = model_path.with_name(model_path.name + ".tmp")
            joblib.dump(final_model, temp_model, compress=3)
            temp_model.replace(model_path)
        save_json(final, final_checkpoint)
    summary["Final_Tuning_Best_CV_RMSE_not_independent"] = final["final_tuning_best_CV_RMSE"]
    params_row = {"LST": target, "Buffer_m": distance, "n_samples": len(y),
                  **final["best_params"], "random_state": cfg.random_state,
                  "bootstrap": True,
                  "Final_Tuning_Best_CV_RMSE_not_independent": final["final_tuning_best_CV_RMSE"]}
    return summary, params_row, folds


# =============================================================================
# 4. 主程序
# =============================================================================
def main(cfg=None):
    cfg = CONFIG if cfg is None else cfg
    started = time.monotonic()
    data, identities, audit, groups, splits, plan, report = prepare_data(cfg)
    root = initialize_output(cfg, report)
    save_csv(audit, root / "sample_inclusion_audit.csv")
    save_csv(identities, root / "common_samples_spatial_folds.csv")
    save_csv(plan, root / "outer_fold_plan.csv")
    save_json(report, root / "sample_report.json")
    combinations = [(d, t) for d in cfg.distances for t in cfg.targets]
    cv_fits = len(combinations) * cfg.n_iter_search * (
        int(plan["inner_folds"].sum()) + min(cfg.inner_folds, len(np.unique(groups)))
    )
    print(f"输入{report['n_input']}个社区；面积>{cfg.minimum_area:g}："
          f"{report['n_area_eligible']}个；共同完整样本：{len(data)}个。", flush=True)
    print(f"{len(np.unique(groups))}个空间块，{len(combinations)}个目标×尺度组合；"
          f"首次完整运行预计{cv_fits:,}次CV拟合，另加每次搜索的refit。", flush=True)
    print(f"输出目录：{root}", flush=True)
    if cfg.dry_run:
        print("dry_run=True：数据/分组审计完成，未训练任何模型。", flush=True)
        return pd.DataFrame(), pd.DataFrame()
    summaries, parameters, all_folds = [], [], []
    for number, (distance, target) in enumerate(combinations, 1):
        print(f"\n组合{number}/{len(combinations)}：{target}, {distance}m", flush=True)
        summary, params, folds = run_combination(
            data, identities, groups, splits, target, distance, root, cfg
        )
        summaries.append(summary)
        parameters.append(params)
        all_folds.extend(folds)
        # 每完成一个组合立即更新汇总，后续报错也不会丢失已完成结果。
        results_df = pd.DataFrame(summaries)
        params_df = pd.DataFrame(parameters)
        save_csv(results_df, root / "RF_multiscale_spatial_cv_metrics.csv")
        save_csv(params_df, root / "RF_multiscale_final_best_params.csv")
        save_csv(pd.DataFrame(all_folds), root / "RF_all_outer_fold_metrics_and_params.csv")
        print(f"外层OOF R²={summary['Spatial_CV_R2']:.4f}，"
              f"RMSE={summary['Spatial_CV_RMSE']:.4f}；"
              f"本次已用时{(time.monotonic()-started)/60:.1f}分钟", flush=True)

    if cfg.export_excel:
        output_xlsx = root / "RF_multiscale_spatial_cv_results.xlsx"
        temp_xlsx = root / "RF_multiscale_spatial_cv_results.tmp.xlsx"
        try:
            with pd.ExcelWriter(temp_xlsx, engine="openpyxl") as writer:
                results_df.to_excel(writer, sheet_name="Nested_OOF_metrics", index=False)
                params_df.to_excel(writer, sheet_name="Final_best_params", index=False)
                pd.DataFrame(all_folds).to_excel(writer, sheet_name="Outer_fold_results", index=False)
                identities.to_excel(writer, sheet_name="Common_sample_folds", index=False)
                audit.to_excel(writer, sheet_name="Sample_inclusion_audit", index=False)
            temp_xlsx.replace(output_xlsx)
        except (PermissionError, ImportError) as exc:
            print(f"Excel汇总未成功写入（CSV及模型已保存）：{exc}。"
                  "关闭Excel占用或安装openpyxl后重新运行，可从断点恢复。", flush=True)
    print("\n完成。用于论文性能报告的是Spatial_CV_R2/Spatial_CV_RMSE，"
          "不是Final_Tuning_Best_CV_RMSE_not_independent。", flush=True)
    print(results_df.to_string(index=False))
    return results_df, params_df


if __name__ == "__main__":
    main()


输入949个社区；面积>10000：949个；共同完整样本：949个。
370个空间块，20个目标×尺度组合；首次完整运行预计30,000次CV拟合，另加每次搜索的refit。
输出目录：E:\博士申请\第二篇论文\脚本\Python\importance\multiscale_spatial_cv_area_gt10000_ISF

组合1/20：SUNNY, 30m
[SUNNY, 30m] 开始外层折1/5，内层5折×50组参数
Fitting 5 folds for each of 50 candidates, totalling 250 fits


In [ ]:
# -*- coding: utf-8 -*-
"""Area > 10000 m²: 三个指定模型的随机森林嵌套空间交叉验证及最终调优。

直接在 Jupyter 中运行本文件（%run "完整路径.py"）或复制全文到单元格。
不使用 argparse，不会出现 ipykernel 的 -f 参数错误。

方法与原脚本一致：1000 m 方形网格、外层 GroupKFold 评价、内层
GroupKFold 调参。模型组合：lst0603–90 m、lst0619–90 m、D_LST–300 m。
保留共同完整样本、逐折保存、
断点续跑、全样本最终调参和模型导出。坐标必须为 EPSG:32650，单位米。

注意：空间块不同不代表训练/测试点之间存在最小隔离距离；本脚本没有
额外的空间排除带，亦不保证完全消除空间相关性。不要把 90/300 m 的
景观缓冲距离与 1000 m 的空间验证块大小混为一谈。

D_LST=0及负值均保留；仅lst0603/lst0619的0值延续原脚本的缺失编码。
内部RCI、外部ISF沿用原脚本，面积筛选仍为Shape_Area>10000。

依赖：numpy pandas scipy scikit-learn xlrd openpyxl joblib。
"""

from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import hashlib
import json
import platform
import time

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, RandomizedSearchCV


# =============================================================================
# 1. 用户设置（默认正式运行；不需要修改后面的函数）
# =============================================================================
@dataclass
class Config:
    data_path: str = (
        r"E:\博士申请\第二篇论文\data\wuhuanshiliang\SH_attributes_10000.xls"
    )
    # 独立新文件夹：不覆盖此前的随机划分验证/SHAP结果。
    output_dir: str = (
        r"E:\博士申请\第二篇论文\脚本\Python\importance"
        r"\three_models_spatial_cv_area_gt10000_ISF"
    )
    sheet_name: int = 0
    area_column: str = "Shape_Area"  # 实际面积字段，不是字面上的 AREA
    minimum_area: float = 10000.0   # 严格大于；恰好10000也不纳入
    # 显式指定三个配对，不能展开成目标×距离的笛卡尔积。
    model_specs: tuple = (("lst0603", 90), ("lst0619", 90), ("D_LST", 300))
    zero_as_missing_targets: tuple = ("lst0603", "lst0619")
    coordinate_columns: tuple = ("CENT_X", "CENT_Y")
    fixed_vars: tuple = (
        "TCC", "GCI", "BCI", "WCI", "RCI", "Shape_Area",
        "jungong", "BAH", "BHSD", "DIST",
    )
    buffer_prefixes: tuple = ("TCC", "GCI", "BCI", "WCI", "ISF", "BAH", "BHSD")
    random_state: int = 123
    block_size: float = 1000.0
    outer_folds: int = 5
    inner_folds: int = 5
    n_iter_search: int = 50
    # 保留原脚本的参数搜索范围；这里上下界均包含。
    n_estimators_min: int = 500
    n_estimators_max: int = 2000
    max_features_min: int = 3
    min_samples_leaf_min: int = 1
    min_samples_leaf_max: int = 15
    # 只在搜索层并行；森林内部单线程，避免两层同时 n_jobs=-1。
    search_jobs: int = 4         # 内存不足时改为2或1；-1为全部CPU
    search_verbose: int = 1
    save_final_models: bool = True
    export_excel: bool = True
    resume: bool = True         # 相同数据和设置时，跳过已完成的折/最终调优
    dry_run: bool = False       # True只检查数据、空间折并输出审计文件，不训练

    @property
    def targets(self):
        return tuple(dict.fromkeys(target for target, _ in self.model_specs))


CONFIG = Config()
SCRIPT_VERSION = "three-models-1.0.0"


# =============================================================================
# 2. 通用保存、读入、审计
# =============================================================================
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value)}")


def save_json(value, path):
    """先写临时文件再替换：只把完整结果视为断点。"""
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                               default=json_default), encoding="utf-8")
    temp.replace(path)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def save_csv(frame, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8-sig")
    temp.replace(path)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def features_for_distance(distance, cfg):
    # 同一模型只纳入该距离的7个外部变量，不同时纳入其他距离。
    return list(cfg.fixed_vars) + [f"{v}_b{distance}m" for v in cfg.buffer_prefixes]


def prepare_data(cfg):
    path = Path(cfg.data_path)
    if not path.is_file():
        raise FileNotFoundError(f"找不到数据文件，请修改 CONFIG.data_path：{path}")
    if cfg.block_size <= 0 or min(cfg.outer_folds, cfg.inner_folds) < 2:
        raise ValueError("空间块边长须>0，内外层折数须>=2。")
    if not cfg.model_specs or len(set(cfg.model_specs)) != len(cfg.model_specs):
        raise ValueError("model_specs不能为空或包含重复模型组合。")
    for target, distance in cfg.model_specs:
        if not isinstance(target, str) or not target.strip():
            raise ValueError("每个模型须指定有效的因变量名称。")
        if not isinstance(distance, int) or distance <= 0:
            raise ValueError("每个模型须指定正整数缓冲距离。")
    if not set(cfg.zero_as_missing_targets).issubset(cfg.targets):
        raise ValueError("zero_as_missing_targets须属于model_specs中的因变量。")
    engine = "xlrd" if path.suffix.lower() == ".xls" else "openpyxl"
    raw = pd.read_excel(path, sheet_name=cfg.sheet_name, engine=engine)
    required = list(dict.fromkeys(
        [cfg.area_column] + list(cfg.targets) + list(cfg.coordinate_columns)
        + [v for _, d in cfg.model_specs for v in features_for_distance(d, cfg)]
    ))
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise KeyError("缺少必需字段：\n" + "\n".join(missing))
    numeric = raw[required].apply(pd.to_numeric, errors="coerce")
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    for target in cfg.zero_as_missing_targets:
        numeric.loc[numeric[target] == 0, target] = np.nan
    # D_LST是温差，0值和负值有物理意义，不能按缺失值处理。

    area_ok = numeric[cfg.area_column] > cfg.minimum_area
    complete = numeric.notna().all(axis=1)
    included = area_ok & complete
    # 用Excel行号作唯一追踪键；不依据可能重复的社区名称拼接结果。
    audit = pd.DataFrame({"source_excel_row": np.arange(len(raw)) + 2})
    for name in ("name", "NAME", "FID", "OBJECTID"):
        if name in raw.columns:
            audit[name] = raw[name].to_numpy()
    audit[cfg.area_column] = numeric[cfg.area_column].to_numpy()
    audit["area_gt_minimum"] = area_ok.to_numpy()
    audit["complete_all_selected_models"] = complete.to_numpy()
    audit["included"] = included.to_numpy()
    audit["invalid_or_missing_columns"] = numeric.isna().apply(
        lambda row: ";".join(row.index[row]), axis=1
    )
    audit["exclusion_reason"] = np.select(
        [numeric[cfg.area_column].isna(), ~area_ok, ~complete],
        ["missing_or_invalid_area", "area_not_greater_than_minimum",
         "incomplete_required_columns"], default="included"
    )
    data = numeric.loc[included].reset_index(drop=True)
    identities = audit.loc[included].reset_index(drop=True)
    if len(data) < 50:
        raise ValueError(f"共同完整案例只有{len(data)}个，少于50。")
    for target in cfg.targets:
        if data[target].nunique() < 2:
            raise ValueError(f"{target}无变异，无法评价R²。")
    xy = data[list(cfg.coordinate_columns)].to_numpy(dtype=float)
    if np.max(np.abs(xy[:, 0])) <= 180 and np.max(np.abs(xy[:, 1])) <= 90:
        raise ValueError("坐标看起来是经纬度；须先转换到以米为单位的EPSG:32650。")
    origin = xy.min(axis=0)
    block_xy = np.floor((xy - origin) / cfg.block_size).astype(np.int64)
    labels = pd.Series(list(map(tuple, block_xy)))
    groups = pd.factorize(labels, sort=True)[0]
    if len(np.unique(groups)) < cfg.outer_folds:
        raise ValueError("有效空间组数量少于外层折数。")
    outer_splits = list(GroupKFold(n_splits=cfg.outer_folds).split(data, groups=groups))
    fold_assignment = np.zeros(len(data), dtype=int)
    fold_plan = []
    for fold, (train, test) in enumerate(outer_splits, 1):
        assert not np.intersect1d(groups[train], groups[test]).size
        n_inner = min(cfg.inner_folds, len(np.unique(groups[train])))
        if n_inner < 2 or len(test) < 2:
            raise ValueError("空间组/样本不足，不能执行指定的嵌套验证。")
        fold_assignment[test] = fold
        fold_plan.append({"outer_fold": fold, "n_train": len(train),
                          "n_test": len(test), "n_train_blocks": len(np.unique(groups[train])),
                          "n_test_blocks": len(np.unique(groups[test])), "inner_folds": n_inner})
    identities["CENT_X"] = xy[:, 0]
    identities["CENT_Y"] = xy[:, 1]
    identities["block_x"] = block_xy[:, 0]
    identities["block_y"] = block_xy[:, 1]
    identities["spatial_group"] = groups
    identities["outer_fold"] = fold_assignment
    report = {
        "n_input": len(raw), "n_area_eligible": int(area_ok.sum()),
        "n_common_complete": len(data), "n_spatial_blocks": len(np.unique(groups)),
        "n_excluded_area_or_missing_area": int((~area_ok).sum()),
        "n_excluded_incomplete_after_area_filter": int((area_ok & ~complete).sum()),
        "area_column": cfg.area_column, "minimum_area_exclusive_m2": cfg.minimum_area,
        "grid_origin_x": float(origin[0]), "grid_origin_y": float(origin[1]),
        "block_size_m": cfg.block_size,
        "common_sample_rule": "Complete for all three selected models and coordinates",
        "model_specs": cfg.model_specs,
        "zero_as_missing_targets": cfg.zero_as_missing_targets,
    }
    return data, identities, audit, groups, outer_splits, pd.DataFrame(fold_plan), report


def initialize_output(cfg, report):
    root = Path(cfg.output_dir)
    root.mkdir(parents=True, exist_ok=True)
    settings = asdict(cfg)
    # 仅这些运行控制项改变时，可以继续复用同一计算结果。
    for key in ("output_dir", "resume", "dry_run", "search_jobs", "search_verbose", "export_excel"):
        settings.pop(key)
    versions = {"python": platform.python_version(), "numpy": np.__version__,
                "pandas": pd.__version__, "scipy": scipy.__version__,
                "sklearn": sklearn.__version__, "joblib": joblib.__version__}
    signature_inputs = {"script_version": SCRIPT_VERSION, "settings": settings,
                        "input_sha256": file_sha256(cfg.data_path), "versions": versions}
    signature = hashlib.sha256(json.dumps(signature_inputs, sort_keys=True,
                                          default=json_default).encode("utf-8")).hexdigest()
    manifest_path = root / "run_metadata.json"
    if manifest_path.exists():
        old = load_json(manifest_path)
        if old["signature"] != signature:
            raise ValueError("输出目录存在不同数据/参数/版本的结果。请更换 output_dir，避免混用。")
        if not cfg.resume:
            raise ValueError("目录已有本次结果；使用resume=True续跑，或换新输出目录重跑。")
    else:
        if any(root.iterdir()):
            raise ValueError("输出目录非空且没有运行记录。请为本脚本选择一个新的子文件夹。")
        save_json({"signature": signature, **signature_inputs, "sample_report": report,
                   "created_utc": datetime.now(timezone.utc).isoformat()}, manifest_path)
    return root


# =============================================================================
# 3. 空间调参：三个模型使用相同的候选参数抽样、随机种子和空间划分
# =============================================================================
def build_search(n_features, n_splits, cfg):
    if cfg.n_iter_search < 1 or cfg.search_jobs == 0:
        raise ValueError("n_iter_search须>=1，search_jobs不能为0。")
    for low, high in ((cfg.n_estimators_min, cfg.n_estimators_max),
                      (cfg.min_samples_leaf_min, cfg.min_samples_leaf_max),
                      (cfg.max_features_min, n_features)):
        if low < 1 or low > high:
            raise ValueError("参数搜索上下界无效。")
    return RandomizedSearchCV(
        estimator=RandomForestRegressor(bootstrap=True, random_state=cfg.random_state, n_jobs=1),
        param_distributions={
            "n_estimators": randint(cfg.n_estimators_min, cfg.n_estimators_max + 1),
            "max_features": randint(cfg.max_features_min, n_features + 1),
            "min_samples_leaf": randint(cfg.min_samples_leaf_min, cfg.min_samples_leaf_max + 1),
        },
        n_iter=cfg.n_iter_search, scoring="neg_root_mean_squared_error",
        cv=GroupKFold(n_splits=n_splits), random_state=cfg.random_state,
        n_jobs=cfg.search_jobs, pre_dispatch="n_jobs", verbose=cfg.search_verbose,
        refit=True, error_score="raise", return_train_score=False,
    )


def metrics(observed, predicted):
    return {
        "R2": float(r2_score(observed, predicted)) if len(observed) > 1 and np.var(observed) > 0 else None,
        "RMSE": float(np.sqrt(mean_squared_error(observed, predicted))),
    }


def save_search_results(search, path):
    results = pd.DataFrame(search.cv_results_)
    results["mean_validation_RMSE"] = -results["mean_test_score"]
    results["std_validation_RMSE"] = results["std_test_score"]
    save_csv(results, path)


def run_combination(data, identities, groups, outer_splits, target, distance, root, cfg):
    folder = root / f"buffer_{distance:03d}m" / target
    folder.mkdir(parents=True, exist_ok=True)
    features = features_for_distance(distance, cfg)
    X = data[features]
    y = data[target].to_numpy(dtype=float)
    predictions = np.full(len(y), np.nan)
    folds = []
    for fold, (train, test) in enumerate(outer_splits, 1):
        fold_dir = folder / f"outer_fold_{fold:02d}"
        fold_dir.mkdir(exist_ok=True)
        checkpoint = fold_dir / "completed.json"
        if checkpoint.exists() and cfg.resume:
            record = load_json(checkpoint)
            if not np.array_equal(record["test_indices"], test):
                raise RuntimeError("断点测试索引与当前空间折不一致。")
            print(f"[{target}, {distance}m] 外层折{fold}：读取已完成断点", flush=True)
        else:
            n_inner = min(cfg.inner_folds, len(np.unique(groups[train])))
            print(f"[{target}, {distance}m] 开始外层折{fold}/{len(outer_splits)}，"
                  f"内层{n_inner}折×{cfg.n_iter_search}组参数", flush=True)
            started = time.monotonic()
            search = build_search(len(features), n_inner, cfg)
            search.fit(X.iloc[train], y[train], groups=groups[train])
            predicted = search.best_estimator_.predict(X.iloc[test])
            scores = metrics(y[test], predicted)
            record = {
                "LST": target, "Buffer_m": distance, "outer_fold": fold,
                "n_train": len(train), "n_test": len(test),
                "n_train_blocks": len(np.unique(groups[train])),
                "n_test_blocks": len(np.unique(groups[test])), "inner_folds": n_inner,
                "test_R2": scores["R2"], "test_RMSE": scores["RMSE"],
                "inner_best_CV_RMSE": float(-search.best_score_),
                "best_params": search.best_params_,
                "elapsed_seconds": time.monotonic() - started,
                "test_indices": test.tolist(), "predictions": predicted.tolist(),
            }
            save_search_results(search, fold_dir / "inner_search_results.csv")
            # 最后写completed.json：如果中途退出，只重跑尚未完成的一折。
            save_json(record, checkpoint)
            print(f"  完成：外层RMSE={scores['RMSE']:.4f}，"
                  f"用时{record['elapsed_seconds']/60:.1f}分钟", flush=True)
        predicted = np.asarray(record["predictions"], dtype=float)
        if len(predicted) != len(test) or not np.isfinite(predicted).all():
            raise RuntimeError(f"断点预测无效：{checkpoint}")
        predictions[test] = predicted
        folds.append({k: v for k, v in record.items()
                      if k not in ("test_indices", "predictions", "best_params")} | record["best_params"])
    if not np.isfinite(predictions).all():
        raise RuntimeError("存在未获得外层OOF预测的社区。")
    pooled = metrics(y, predictions)
    prediction_table = identities.copy()
    prediction_table["LST"] = target
    prediction_table["Buffer_m"] = distance
    prediction_table["observed_LST"] = y
    prediction_table["predicted_LST_OOF"] = predictions
    prediction_table["residual_observed_minus_predicted"] = y - predictions
    save_csv(prediction_table, folder / "outer_oof_predictions.csv")
    save_csv(pd.DataFrame(folds), folder / "outer_fold_metrics_and_params.csv")
    summary = {
        "LST": target, "Buffer_m": distance, "n_samples": len(y),
        "n_spatial_blocks": len(np.unique(groups)),
        "Spatial_CV_R2": pooled["R2"], "Spatial_CV_RMSE": pooled["RMSE"],
        "outer_folds": len(outer_splits), "Block_size_m": cfg.block_size,
    }
    save_json(summary, folder / "nested_spatial_cv_metrics.json")

    # 独立阶段：在全部共同样本中进行空间CV调参、全样本refit，供后续SHAP/情景模拟使用。
    # 不能用此阶段的调参最优分数替代外层OOF性能。
    final_checkpoint = folder / "final_tuning_completed.json"
    model_path = folder / "final_rf_model.joblib"
    if final_checkpoint.exists() and cfg.resume:
        final = load_json(final_checkpoint)
        if cfg.save_final_models and not model_path.is_file():
            raise FileNotFoundError(f"最终模型丢失：{model_path}；请换新输出目录重新运行。")
        print(f"[{target}, {distance}m] 最终调优：读取已完成断点", flush=True)
    else:
        n_final_folds = min(cfg.inner_folds, len(np.unique(groups)))
        print(f"[{target}, {distance}m] 开始全样本空间调优（用于最终参数，不用于独立评价）", flush=True)
        started = time.monotonic()
        final_search = build_search(len(features), n_final_folds, cfg)
        final_search.fit(X, y, groups=groups)
        final_model = final_search.best_estimator_
        save_search_results(final_search, folder / "final_search_results.csv")
        final = {
            "LST": target, "Buffer_m": distance, "n_samples": len(y),
            "features_in_order": features, "best_params": final_search.best_params_,
            "all_rf_params": final_model.get_params(),
            "final_tuning_best_CV_RMSE": float(-final_search.best_score_),
            "final_tuning_folds": n_final_folds, "elapsed_seconds": time.monotonic() - started,
            "score_note": "Selection score, NOT independent performance; use nested OOF metrics.",
        }
        save_json(final, folder / "final_best_params.json")
        if cfg.save_final_models:
            temp_model = model_path.with_name(model_path.name + ".tmp")
            joblib.dump(final_model, temp_model, compress=3)
            temp_model.replace(model_path)
        save_json(final, final_checkpoint)
    summary["Final_Tuning_Best_CV_RMSE_not_independent"] = final["final_tuning_best_CV_RMSE"]
    params_row = {"LST": target, "Buffer_m": distance, "n_samples": len(y),
                  **final["best_params"], "random_state": cfg.random_state,
                  "bootstrap": True,
                  "Final_Tuning_Best_CV_RMSE_not_independent": final["final_tuning_best_CV_RMSE"]}
    return summary, params_row, folds


# =============================================================================
# 4. 主程序
# =============================================================================
def main(cfg=None):
    cfg = CONFIG if cfg is None else cfg
    started = time.monotonic()
    data, identities, audit, groups, splits, plan, report = prepare_data(cfg)
    root = initialize_output(cfg, report)
    save_csv(audit, root / "sample_inclusion_audit.csv")
    save_csv(identities, root / "common_samples_spatial_folds.csv")
    save_csv(plan, root / "outer_fold_plan.csv")
    save_json(report, root / "sample_report.json")
    combinations = [(distance, target) for target, distance in cfg.model_specs]
    cv_fits = len(combinations) * cfg.n_iter_search * (
        int(plan["inner_folds"].sum()) + min(cfg.inner_folds, len(np.unique(groups)))
    )
    print(f"输入{report['n_input']}个社区；面积>{cfg.minimum_area:g}："
          f"{report['n_area_eligible']}个；共同完整样本：{len(data)}个。", flush=True)
    print(f"{len(np.unique(groups))}个空间块，{len(combinations)}个指定模型组合；"
          f"首次完整运行预计{cv_fits:,}次CV拟合，另加每次搜索的refit。", flush=True)
    print("指定模型：" + "；".join(f"{t}–{d} m" for t, d in cfg.model_specs), flush=True)
    print(f"输出目录：{root}", flush=True)
    if cfg.dry_run:
        print("dry_run=True：数据/分组审计完成，未训练任何模型。", flush=True)
        return pd.DataFrame(), pd.DataFrame()
    summaries, parameters, all_folds = [], [], []
    for number, (distance, target) in enumerate(combinations, 1):
        print(f"\n组合{number}/{len(combinations)}：{target}, {distance}m", flush=True)
        summary, params, folds = run_combination(
            data, identities, groups, splits, target, distance, root, cfg
        )
        summaries.append(summary)
        parameters.append(params)
        all_folds.extend(folds)
        # 每完成一个组合立即更新汇总，后续报错也不会丢失已完成结果。
        results_df = pd.DataFrame(summaries)
        params_df = pd.DataFrame(parameters)
        save_csv(results_df, root / "RF_three_models_spatial_cv_metrics.csv")
        save_csv(params_df, root / "RF_three_models_final_best_params.csv")
        save_csv(pd.DataFrame(all_folds), root / "RF_all_outer_fold_metrics_and_params.csv")
        print(f"外层OOF R²={summary['Spatial_CV_R2']:.4f}，"
              f"RMSE={summary['Spatial_CV_RMSE']:.4f}；"
              f"本次已用时{(time.monotonic()-started)/60:.1f}分钟", flush=True)

    if cfg.export_excel:
        output_xlsx = root / "RF_three_models_spatial_cv_results.xlsx"
        temp_xlsx = root / "RF_three_models_spatial_cv_results.tmp.xlsx"
        try:
            with pd.ExcelWriter(temp_xlsx, engine="openpyxl") as writer:
                results_df.to_excel(writer, sheet_name="Nested_OOF_metrics", index=False)
                params_df.to_excel(writer, sheet_name="Final_best_params", index=False)
                pd.DataFrame(all_folds).to_excel(writer, sheet_name="Outer_fold_results", index=False)
                identities.to_excel(writer, sheet_name="Common_sample_folds", index=False)
                audit.to_excel(writer, sheet_name="Sample_inclusion_audit", index=False)
            temp_xlsx.replace(output_xlsx)
        except (PermissionError, ImportError) as exc:
            print(f"Excel汇总未成功写入（CSV及模型已保存）：{exc}。"
                  "关闭Excel占用或安装openpyxl后重新运行，可从断点恢复。", flush=True)
    print("\n完成。用于论文性能报告的是Spatial_CV_R2/Spatial_CV_RMSE，"
          "不是Final_Tuning_Best_CV_RMSE_not_independent。", flush=True)
    print(results_df.to_string(index=False))
    return results_df, params_df


if __name__ == "__main__":
    main()


输入949个社区；面积>10000：949个；共同完整样本：949个。
370个空间块，3个指定模型组合；首次完整运行预计4,500次CV拟合，另加每次搜索的refit。
指定模型：lst0603–90 m；lst0619–90 m；D_LST–300 m
输出目录：E:\博士申请\第二篇论文\脚本\Python\importance\three_models_spatial_cv_area_gt10000_ISF

组合1/3：lst0603, 90m
[lst0603, 90m] 开始外层折1/5，内层5折×50组参数
Fitting 5 folds for each of 50 candidates, totalling 250 fits


In [2]:
# -*- coding: utf-8 -*-
"""Area > 10000 m²: 三个指定模型的随机森林嵌套空间交叉验证及最终调优。

直接在 Jupyter 中运行本文件（%run "完整路径.py"）或复制全文到单元格。
不使用 argparse，不会出现 ipykernel 的 -f 参数错误。

方法与原脚本一致：1000 m 方形网格、外层 GroupKFold 评价、内层
GroupKFold 调参。模型组合：lst0603–90 m、lst0619–90 m、D_LST–300 m。
保留共同完整样本、逐折保存、
断点续跑、全样本最终调参和模型导出。坐标必须为 EPSG:32650，单位米。

注意：空间块不同不代表训练/测试点之间存在最小隔离距离；本脚本没有
额外的空间排除带，亦不保证完全消除空间相关性。不要把 90/300 m 的
景观缓冲距离与 1000 m 的空间验证块大小混为一谈。

D_LST=0及负值均保留；仅lst0603/lst0619的0值延续原脚本的缺失编码。
内部RCI、外部ISF沿用原脚本，面积筛选仍为Shape_Area>10000。

依赖：numpy pandas scipy scikit-learn xlrd openpyxl joblib。
"""

from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import hashlib
import json
import platform
import time

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, RandomizedSearchCV


# =============================================================================
# 1. 用户设置（默认正式运行；不需要修改后面的函数）
# =============================================================================
@dataclass
class Config:
    data_path: str = (
        r"E:\博士申请\第二篇论文\data\wuhuanshiliang\SH_attributes_10000.xls"
    )
    # 独立新文件夹：不覆盖此前的随机划分验证/SHAP结果。
    output_dir: str = (
        r"E:\博士申请\第二篇论文\脚本\Python\importance"
        r"\three_models_spatial_cv_area_gt10000_ISF"
    )
    sheet_name: int = 0
    area_column: str = "Shape_Area"  # 实际面积字段，不是字面上的 AREA
    minimum_area: float = 10000.0   # 严格大于；恰好10000也不纳入
    # 显式指定三个配对，不能展开成目标×距离的笛卡尔积。
    model_specs: tuple = (("lst0603", 90), ("lst0619", 90), ("D_LST", 300))
    zero_as_missing_targets: tuple = ("lst0603", "lst0619")
    coordinate_columns: tuple = ("CENT_X", "CENT_Y")
    fixed_vars: tuple = (
        "TCC", "GCI", "BCI", "WCI", "RCI", "Shape_Area",
        "jungong", "BAH", "BHSD", "DIST",
    )
    buffer_prefixes: tuple = ("TCC", "GCI", "BCI", "WCI", "ISF", "BAH", "BHSD")
    random_state: int = 123
    block_size: float = 1000.0
    outer_folds: int = 5
    inner_folds: int = 5
    n_iter_search: int = 50
    # 保留原脚本的参数搜索范围；这里上下界均包含。
    n_estimators_min: int = 500
    n_estimators_max: int = 2000
    max_features_min: int = 3
    min_samples_leaf_min: int = 1
    min_samples_leaf_max: int = 15
    # 只在搜索层并行；森林内部单线程，避免两层同时 n_jobs=-1。
    search_jobs: int = 4         # 内存不足时改为2或1；-1为全部CPU
    search_verbose: int = 1
    save_final_models: bool = True
    export_excel: bool = True
    resume: bool = True         # 相同数据和设置时，跳过已完成的折/最终调优
    run_final_tuning: bool = False  # 块大小敏感性主任务不需要再次导出全样本最终模型
    dry_run: bool = False       # True只检查数据、空间折并输出审计文件，不训练

    @property
    def targets(self):
        return tuple(dict.fromkeys(target for target, _ in self.model_specs))


CONFIG = Config()
SCRIPT_VERSION = "three-models-1.0.0"


# =============================================================================
# 2. 通用保存、读入、审计
# =============================================================================
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value)}")


def save_json(value, path):
    """先写临时文件再替换：只把完整结果视为断点。"""
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                               default=json_default), encoding="utf-8")
    temp.replace(path)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def save_csv(frame, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8-sig")
    temp.replace(path)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def features_for_distance(distance, cfg):
    # 同一模型只纳入该距离的7个外部变量，不同时纳入其他距离。
    return list(cfg.fixed_vars) + [f"{v}_b{distance}m" for v in cfg.buffer_prefixes]


def prepare_data(cfg):
    path = Path(cfg.data_path)
    if not path.is_file():
        raise FileNotFoundError(f"找不到数据文件，请修改 CONFIG.data_path：{path}")
    if cfg.block_size <= 0 or min(cfg.outer_folds, cfg.inner_folds) < 2:
        raise ValueError("空间块边长须>0，内外层折数须>=2。")
    if not cfg.model_specs or len(set(cfg.model_specs)) != len(cfg.model_specs):
        raise ValueError("model_specs不能为空或包含重复模型组合。")
    for target, distance in cfg.model_specs:
        if not isinstance(target, str) or not target.strip():
            raise ValueError("每个模型须指定有效的因变量名称。")
        if not isinstance(distance, int) or distance <= 0:
            raise ValueError("每个模型须指定正整数缓冲距离。")
    if not set(cfg.zero_as_missing_targets).issubset(cfg.targets):
        raise ValueError("zero_as_missing_targets须属于model_specs中的因变量。")
    engine = "xlrd" if path.suffix.lower() == ".xls" else "openpyxl"
    raw = pd.read_excel(path, sheet_name=cfg.sheet_name, engine=engine)
    required = list(dict.fromkeys(
        [cfg.area_column] + list(cfg.targets) + list(cfg.coordinate_columns)
        + [v for _, d in cfg.model_specs for v in features_for_distance(d, cfg)]
    ))
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise KeyError("缺少必需字段：\n" + "\n".join(missing))
    numeric = raw[required].apply(pd.to_numeric, errors="coerce")
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    for target in cfg.zero_as_missing_targets:
        numeric.loc[numeric[target] == 0, target] = np.nan
    # D_LST是温差，0值和负值有物理意义，不能按缺失值处理。

    area_ok = numeric[cfg.area_column] > cfg.minimum_area
    complete = numeric.notna().all(axis=1)
    included = area_ok & complete
    # 用Excel行号作唯一追踪键；不依据可能重复的社区名称拼接结果。
    audit = pd.DataFrame({"source_excel_row": np.arange(len(raw)) + 2})
    for name in ("name", "NAME", "FID", "OBJECTID"):
        if name in raw.columns:
            audit[name] = raw[name].to_numpy()
    audit[cfg.area_column] = numeric[cfg.area_column].to_numpy()
    audit["area_gt_minimum"] = area_ok.to_numpy()
    audit["complete_all_selected_models"] = complete.to_numpy()
    audit["included"] = included.to_numpy()
    audit["invalid_or_missing_columns"] = numeric.isna().apply(
        lambda row: ";".join(row.index[row]), axis=1
    )
    audit["exclusion_reason"] = np.select(
        [numeric[cfg.area_column].isna(), ~area_ok, ~complete],
        ["missing_or_invalid_area", "area_not_greater_than_minimum",
         "incomplete_required_columns"], default="included"
    )
    data = numeric.loc[included].reset_index(drop=True)
    identities = audit.loc[included].reset_index(drop=True)
    if len(data) < 50:
        raise ValueError(f"共同完整案例只有{len(data)}个，少于50。")
    for target in cfg.targets:
        if data[target].nunique() < 2:
            raise ValueError(f"{target}无变异，无法评价R²。")
    xy = data[list(cfg.coordinate_columns)].to_numpy(dtype=float)
    if np.max(np.abs(xy[:, 0])) <= 180 and np.max(np.abs(xy[:, 1])) <= 90:
        raise ValueError("坐标看起来是经纬度；须先转换到以米为单位的EPSG:32650。")
    origin = xy.min(axis=0)
    block_xy = np.floor((xy - origin) / cfg.block_size).astype(np.int64)
    labels = pd.Series(list(map(tuple, block_xy)))
    groups = pd.factorize(labels, sort=True)[0]
    if len(np.unique(groups)) < cfg.outer_folds:
        raise ValueError("有效空间组数量少于外层折数。")
    outer_splits = list(GroupKFold(n_splits=cfg.outer_folds).split(data, groups=groups))
    fold_assignment = np.zeros(len(data), dtype=int)
    fold_plan = []
    for fold, (train, test) in enumerate(outer_splits, 1):
        assert not np.intersect1d(groups[train], groups[test]).size
        n_inner = min(cfg.inner_folds, len(np.unique(groups[train])))
        if n_inner < 2 or len(test) < 2:
            raise ValueError("空间组/样本不足，不能执行指定的嵌套验证。")
        fold_assignment[test] = fold
        fold_plan.append({"outer_fold": fold, "n_train": len(train),
                          "n_test": len(test), "n_train_blocks": len(np.unique(groups[train])),
                          "n_test_blocks": len(np.unique(groups[test])), "inner_folds": n_inner})
    identities["CENT_X"] = xy[:, 0]
    identities["CENT_Y"] = xy[:, 1]
    identities["block_x"] = block_xy[:, 0]
    identities["block_y"] = block_xy[:, 1]
    identities["spatial_group"] = groups
    identities["outer_fold"] = fold_assignment
    report = {
        "n_input": len(raw), "n_area_eligible": int(area_ok.sum()),
        "n_common_complete": len(data), "n_spatial_blocks": len(np.unique(groups)),
        "n_excluded_area_or_missing_area": int((~area_ok).sum()),
        "n_excluded_incomplete_after_area_filter": int((area_ok & ~complete).sum()),
        "area_column": cfg.area_column, "minimum_area_exclusive_m2": cfg.minimum_area,
        "grid_origin_x": float(origin[0]), "grid_origin_y": float(origin[1]),
        "block_size_m": cfg.block_size,
        "common_sample_rule": "Complete for all three selected models and coordinates",
        "model_specs": cfg.model_specs,
        "zero_as_missing_targets": cfg.zero_as_missing_targets,
    }
    return data, identities, audit, groups, outer_splits, pd.DataFrame(fold_plan), report


def initialize_output(cfg, report):
    root = Path(cfg.output_dir)
    root.mkdir(parents=True, exist_ok=True)
    settings = asdict(cfg)
    # 仅这些运行控制项改变时，可以继续复用同一计算结果。
    for key in ("output_dir", "resume", "dry_run", "search_jobs", "search_verbose", "export_excel"):
        settings.pop(key)
    versions = {"python": platform.python_version(), "numpy": np.__version__,
                "pandas": pd.__version__, "scipy": scipy.__version__,
                "sklearn": sklearn.__version__, "joblib": joblib.__version__}
    signature_inputs = {"script_version": SCRIPT_VERSION, "settings": settings,
                        "input_sha256": file_sha256(cfg.data_path), "versions": versions}
    signature = hashlib.sha256(json.dumps(signature_inputs, sort_keys=True,
                                          default=json_default).encode("utf-8")).hexdigest()
    manifest_path = root / "run_metadata.json"
    if manifest_path.exists():
        old = load_json(manifest_path)
        if old["signature"] != signature:
            raise ValueError("输出目录存在不同数据/参数/版本的结果。请更换 output_dir，避免混用。")
        if not cfg.resume:
            raise ValueError("目录已有本次结果；使用resume=True续跑，或换新输出目录重跑。")
    else:
        if any(root.iterdir()):
            raise ValueError("输出目录非空且没有运行记录。请为本脚本选择一个新的子文件夹。")
        save_json({"signature": signature, **signature_inputs, "sample_report": report,
                   "created_utc": datetime.now(timezone.utc).isoformat()}, manifest_path)
    return root


# =============================================================================
# 3. 空间调参：三个模型使用相同的候选参数抽样、随机种子和空间划分
# =============================================================================
def build_search(n_features, n_splits, cfg):
    if cfg.n_iter_search < 1 or cfg.search_jobs == 0:
        raise ValueError("n_iter_search须>=1，search_jobs不能为0。")
    for low, high in ((cfg.n_estimators_min, cfg.n_estimators_max),
                      (cfg.min_samples_leaf_min, cfg.min_samples_leaf_max),
                      (cfg.max_features_min, n_features)):
        if low < 1 or low > high:
            raise ValueError("参数搜索上下界无效。")
    return RandomizedSearchCV(
        estimator=RandomForestRegressor(bootstrap=True, random_state=cfg.random_state, n_jobs=1),
        param_distributions={
            "n_estimators": randint(cfg.n_estimators_min, cfg.n_estimators_max + 1),
            "max_features": randint(cfg.max_features_min, n_features + 1),
            "min_samples_leaf": randint(cfg.min_samples_leaf_min, cfg.min_samples_leaf_max + 1),
        },
        n_iter=cfg.n_iter_search, scoring="neg_root_mean_squared_error",
        cv=GroupKFold(n_splits=n_splits), random_state=cfg.random_state,
        n_jobs=cfg.search_jobs, pre_dispatch="n_jobs", verbose=cfg.search_verbose,
        refit=True, error_score="raise", return_train_score=False,
    )


def metrics(observed, predicted):
    return {
        "R2": float(r2_score(observed, predicted)) if len(observed) > 1 and np.var(observed) > 0 else None,
        "RMSE": float(np.sqrt(mean_squared_error(observed, predicted))),
    }


def save_search_results(search, path):
    results = pd.DataFrame(search.cv_results_)
    results["mean_validation_RMSE"] = -results["mean_test_score"]
    results["std_validation_RMSE"] = results["std_test_score"]
    save_csv(results, path)


def run_combination(data, identities, groups, outer_splits, target, distance, root, cfg):
    folder = root / f"buffer_{distance:03d}m" / target
    folder.mkdir(parents=True, exist_ok=True)
    features = features_for_distance(distance, cfg)
    X = data[features]
    y = data[target].to_numpy(dtype=float)
    predictions = np.full(len(y), np.nan)
    folds = []
    for fold, (train, test) in enumerate(outer_splits, 1):
        fold_dir = folder / f"outer_fold_{fold:02d}"
        fold_dir.mkdir(exist_ok=True)
        checkpoint = fold_dir / "completed.json"
        if checkpoint.exists() and cfg.resume:
            record = load_json(checkpoint)
            if not np.array_equal(record["test_indices"], test):
                raise RuntimeError("断点测试索引与当前空间折不一致。")
            print(f"[{target}, {distance}m] 外层折{fold}：读取已完成断点", flush=True)
        else:
            n_inner = min(cfg.inner_folds, len(np.unique(groups[train])))
            print(f"[{target}, {distance}m] 开始外层折{fold}/{len(outer_splits)}，"
                  f"内层{n_inner}折×{cfg.n_iter_search}组参数", flush=True)
            started = time.monotonic()
            search = build_search(len(features), n_inner, cfg)
            search.fit(X.iloc[train], y[train], groups=groups[train])
            predicted = search.best_estimator_.predict(X.iloc[test])
            scores = metrics(y[test], predicted)
            record = {
                "LST": target, "Buffer_m": distance, "outer_fold": fold,
                "n_train": len(train), "n_test": len(test),
                "n_train_blocks": len(np.unique(groups[train])),
                "n_test_blocks": len(np.unique(groups[test])), "inner_folds": n_inner,
                "test_R2": scores["R2"], "test_RMSE": scores["RMSE"],
                "inner_best_CV_RMSE": float(-search.best_score_),
                "best_params": search.best_params_,
                "elapsed_seconds": time.monotonic() - started,
                "test_indices": test.tolist(), "predictions": predicted.tolist(),
            }
            save_search_results(search, fold_dir / "inner_search_results.csv")
            # 最后写completed.json：如果中途退出，只重跑尚未完成的一折。
            save_json(record, checkpoint)
            print(f"  完成：外层RMSE={scores['RMSE']:.4f}，"
                  f"用时{record['elapsed_seconds']/60:.1f}分钟", flush=True)
        predicted = np.asarray(record["predictions"], dtype=float)
        if len(predicted) != len(test) or not np.isfinite(predicted).all():
            raise RuntimeError(f"断点预测无效：{checkpoint}")
        predictions[test] = predicted
        folds.append({k: v for k, v in record.items()
                      if k not in ("test_indices", "predictions", "best_params")} | record["best_params"])
    if not np.isfinite(predictions).all():
        raise RuntimeError("存在未获得外层OOF预测的社区。")
    pooled = metrics(y, predictions)
    prediction_table = identities.copy()
    prediction_table["LST"] = target
    prediction_table["Buffer_m"] = distance
    prediction_table["observed_LST"] = y
    prediction_table["predicted_LST_OOF"] = predictions
    prediction_table["residual_observed_minus_predicted"] = y - predictions
    save_csv(prediction_table, folder / "outer_oof_predictions.csv")
    save_csv(pd.DataFrame(folds), folder / "outer_fold_metrics_and_params.csv")
    summary = {
        "LST": target, "Buffer_m": distance, "n_samples": len(y),
        "n_spatial_blocks": len(np.unique(groups)),
        "Spatial_CV_R2": pooled["R2"], "Spatial_CV_RMSE": pooled["RMSE"],
        "outer_folds": len(outer_splits), "Block_size_m": cfg.block_size,
    }
    save_json(summary, folder / "nested_spatial_cv_metrics.json")

    if not cfg.run_final_tuning:
        return summary, {"LST": target, "Buffer_m": distance, "n_samples": len(y),
                         "Final_tuning_status": "Skipped: block-size performance sensitivity only"}, folds

    # 独立阶段：在全部共同样本中进行空间CV调参、全样本refit，供后续SHAP/情景模拟使用。
    # 不能用此阶段的调参最优分数替代外层OOF性能。
    final_checkpoint = folder / "final_tuning_completed.json"
    model_path = folder / "final_rf_model.joblib"
    if final_checkpoint.exists() and cfg.resume:
        final = load_json(final_checkpoint)
        if cfg.save_final_models and not model_path.is_file():
            raise FileNotFoundError(f"最终模型丢失：{model_path}；请换新输出目录重新运行。")
        print(f"[{target}, {distance}m] 最终调优：读取已完成断点", flush=True)
    else:
        n_final_folds = min(cfg.inner_folds, len(np.unique(groups)))
        print(f"[{target}, {distance}m] 开始全样本空间调优（用于最终参数，不用于独立评价）", flush=True)
        started = time.monotonic()
        final_search = build_search(len(features), n_final_folds, cfg)
        final_search.fit(X, y, groups=groups)
        final_model = final_search.best_estimator_
        save_search_results(final_search, folder / "final_search_results.csv")
        final = {
            "LST": target, "Buffer_m": distance, "n_samples": len(y),
            "features_in_order": features, "best_params": final_search.best_params_,
            "all_rf_params": final_model.get_params(),
            "final_tuning_best_CV_RMSE": float(-final_search.best_score_),
            "final_tuning_folds": n_final_folds, "elapsed_seconds": time.monotonic() - started,
            "score_note": "Selection score, NOT independent performance; use nested OOF metrics.",
        }
        save_json(final, folder / "final_best_params.json")
        if cfg.save_final_models:
            temp_model = model_path.with_name(model_path.name + ".tmp")
            joblib.dump(final_model, temp_model, compress=3)
            temp_model.replace(model_path)
        save_json(final, final_checkpoint)
    summary["Final_Tuning_Best_CV_RMSE_not_independent"] = final["final_tuning_best_CV_RMSE"]
    params_row = {"LST": target, "Buffer_m": distance, "n_samples": len(y),
                  **final["best_params"], "random_state": cfg.random_state,
                  "bootstrap": True,
                  "Final_Tuning_Best_CV_RMSE_not_independent": final["final_tuning_best_CV_RMSE"]}
    return summary, params_row, folds


# =============================================================================
# 4. 主程序
# =============================================================================
def run_single_block(cfg=None):
    cfg = CONFIG if cfg is None else cfg
    started = time.monotonic()
    data, identities, audit, groups, splits, plan, report = prepare_data(cfg)
    root = initialize_output(cfg, report)
    save_csv(audit, root / "sample_inclusion_audit.csv")
    save_csv(identities, root / "common_samples_spatial_folds.csv")
    save_csv(plan, root / "outer_fold_plan.csv")
    save_json(report, root / "sample_report.json")
    combinations = [(distance, target) for target, distance in cfg.model_specs]
    cv_fits = len(combinations) * cfg.n_iter_search * (
        int(plan["inner_folds"].sum()) + (min(cfg.inner_folds, len(np.unique(groups))) if cfg.run_final_tuning else 0)
    )
    print(f"输入{report['n_input']}个社区；面积>{cfg.minimum_area:g}："
          f"{report['n_area_eligible']}个；共同完整样本：{len(data)}个。", flush=True)
    print(f"{len(np.unique(groups))}个空间块，{len(combinations)}个指定模型组合；"
          f"首次完整运行预计{cv_fits:,}次CV拟合，另加每次搜索的refit。", flush=True)
    print("指定模型：" + "；".join(f"{t}–{d} m" for t, d in cfg.model_specs), flush=True)
    print(f"输出目录：{root}", flush=True)
    if cfg.dry_run:
        print("dry_run=True：数据/分组审计完成，未训练任何模型。", flush=True)
        return pd.DataFrame(), pd.DataFrame()
    summaries, parameters, all_folds = [], [], []
    for number, (distance, target) in enumerate(combinations, 1):
        print(f"\n组合{number}/{len(combinations)}：{target}, {distance}m", flush=True)
        summary, params, folds = run_combination(
            data, identities, groups, splits, target, distance, root, cfg
        )
        summaries.append(summary)
        parameters.append(params)
        all_folds.extend(folds)
        # 每完成一个组合立即更新汇总，后续报错也不会丢失已完成结果。
        results_df = pd.DataFrame(summaries)
        params_df = pd.DataFrame(parameters)
        save_csv(results_df, root / "RF_three_models_spatial_cv_metrics.csv")
        save_csv(params_df, root / "RF_three_models_final_best_params.csv")
        save_csv(pd.DataFrame(all_folds), root / "RF_all_outer_fold_metrics_and_params.csv")
        print(f"外层OOF R²={summary['Spatial_CV_R2']:.4f}，"
              f"RMSE={summary['Spatial_CV_RMSE']:.4f}；"
              f"本次已用时{(time.monotonic()-started)/60:.1f}分钟", flush=True)

    if cfg.export_excel:
        output_xlsx = root / "RF_three_models_spatial_cv_results.xlsx"
        temp_xlsx = root / "RF_three_models_spatial_cv_results.tmp.xlsx"
        try:
            with pd.ExcelWriter(temp_xlsx, engine="openpyxl") as writer:
                results_df.to_excel(writer, sheet_name="Nested_OOF_metrics", index=False)
                params_df.to_excel(writer, sheet_name="Final_best_params", index=False)
                pd.DataFrame(all_folds).to_excel(writer, sheet_name="Outer_fold_results", index=False)
                identities.to_excel(writer, sheet_name="Common_sample_folds", index=False)
                audit.to_excel(writer, sheet_name="Sample_inclusion_audit", index=False)
            temp_xlsx.replace(output_xlsx)
        except (PermissionError, ImportError) as exc:
            print(f"Excel汇总未成功写入（CSV及模型已保存）：{exc}。"
                  "关闭Excel占用或安装openpyxl后重新运行，可从断点恢复。", flush=True)
    print("\n完成。用于论文性能报告的是Spatial_CV_R2/Spatial_CV_RMSE，"
          "不是Final_Tuning_Best_CV_RMSE_not_independent。", flush=True)
    print(results_df.to_string(index=False))
    return results_df, params_df





# =============================================================================
# 5. 比较500/1000/1500 m：相同数据、变量、网格原点、种子和参数候选
# =============================================================================
# 网格原点沿用原脚本：共同样本CENT_X、CENT_Y的最小值。
# 尺度变化会改变空间组和折分配，不能将不同块大小的第1折视为配对样本。
# 此处比较的是验证方案敏感性，不自动做t检验，不按最高R²选择“最佳”块大小。
# 默认所有块大小均重新内层调参，但跳过无关的全样本最终调优。
import os
from dataclasses import replace

BLOCK_SIZES_M = (500, 1000, 1500)
OUTPUT_ROOT = Path(os.environ.get(
    "RF_BLOCK_SENS_OUTPUT",
    r"E:\博士申请\第二篇论文\脚本\Python\importance\three_models_block_size_sensitivity_500_1000_1500"
))
DRY_RUN = False  # 首次可改True：仅检查共同样本、块数、各折样本量，不训练。
RUN_FINAL_TUNING = False  # 性能敏感性分析不需要；True会额外调优并导出每种设置的最终模型。
N_ITER_SEARCH = 50
SEARCH_JOBS = 4

def main(base_config=None, block_sizes=None, output_root=None, dry_run=None):
    cfg_base = Config() if base_config is None else base_config
    sizes = BLOCK_SIZES_M if block_sizes is None else tuple(block_sizes)
    if 1000 not in sizes or len(set(sizes)) != len(sizes) or any(b <= 0 for b in sizes):
        raise ValueError("Block sizes must be unique positive values and include 1000 m.")
    root = OUTPUT_ROOT if output_root is None else Path(output_root)
    root.mkdir(parents=True, exist_ok=True)
    dry = DRY_RUN if dry_run is None else bool(dry_run)

    # 锁定本次基准样本：不同空间块大小不改变纳入条件。
    reference_data, reference_ids, _, _, _, _, _ = prepare_data(cfg_base)
    baseline_rows = reference_ids.source_excel_row.to_numpy()
    summaries, fold_tables, plans = [], [], []
    expected_fits = 0
    # 在正式训练前，先检查全部块大小是否能形成有效嵌套空间折。
    for size in sizes:
        cfg = replace(
            cfg_base, block_size=float(size),
            output_dir=str(root / f"block_{int(size):04d}m"),
            dry_run=dry,
        )
        data, identities, _, groups, splits, plan, report = prepare_data(cfg)
        np.testing.assert_array_equal(identities.source_excel_row, baseline_rows)
        pd.testing.assert_frame_equal(data, reference_data)
        if not np.all(plan.inner_folds == cfg.inner_folds):
            raise ValueError("All block sizes must support the same requested inner fold count.")
        plan = plan.assign(Block_size_m=size, n_spatial_blocks=report["n_spatial_blocks"])
        plans.append(plan)
        expected_fits += len(cfg.model_specs) * (
            cfg.n_iter_search * int(plan.inner_folds.sum()) + len(plan)
            + ((cfg.n_iter_search * cfg.inner_folds + 1) if cfg.run_final_tuning else 0)
        )
    save_csv(pd.concat(plans, ignore_index=True), root / "all_block_outer_fold_plan.csv")
    print(f"共同样本={len(reference_data)}；块大小={sizes}", flush=True)
    print(f"首次完整拟合次数（包括每次搜索的refit）约={expected_fits:,}", flush=True)
    print(f"不同块大小均使用相同的{cfg_base.n_iter_search}组候选参数；只在各外层训练集内选择最优参数。", flush=True)
    for size in sizes:
        # 各块子目录保留CSV和JSON断点；仅在根目录统一输出一个Excel，避免重复工作簿。
        cfg = replace(cfg_base, block_size=float(size), export_excel=False,
                      output_dir=str(root / f"block_{int(size):04d}m"), dry_run=dry)
        scores, _ = run_single_block(cfg)
        if dry:
            continue
        summaries.append(scores)
        folds = pd.read_csv(Path(cfg.output_dir) / "RF_all_outer_fold_metrics_and_params.csv")
        folds.insert(0, "Block_size_m", size)
        fold_tables.append(folds)
        # 每完成一种块大小就更新阶段结果，中断后重跑读取原有逐折断点。
        save_csv(pd.concat(summaries, ignore_index=True), root / "spatial_block_sensitivity_partial.csv")
    if dry:
        print("预检查完成，未训练任何模型。", flush=True)
        return pd.DataFrame()

    combined = pd.concat(summaries, ignore_index=True)
    all_folds = pd.concat(fold_tables, ignore_index=True)
    baseline = combined.loc[combined.Block_size_m.eq(1000),
        ["LST", "Buffer_m", "Spatial_CV_R2", "Spatial_CV_RMSE"]].rename(columns={
            "Spatial_CV_R2": "R2_1000m_reference",
            "Spatial_CV_RMSE": "RMSE_1000m_reference",
        })
    comparison = combined.merge(baseline, on=["LST", "Buffer_m"], how="left", validate="many_to_one")
    comparison["R2_difference_vs_1000m"] = comparison.Spatial_CV_R2 - comparison.R2_1000m_reference
    comparison["RMSE_difference_vs_1000m"] = comparison.Spatial_CV_RMSE - comparison.RMSE_1000m_reference
    comparison["RMSE_relative_difference_pct"] = (
        comparison.RMSE_difference_vs_1000m / comparison.RMSE_1000m_reference * 100
    )
    # 折间标准差只描述折间变动，不是置信区间；主要指标仍是合并OOF预测的指标。
    dispersion = all_folds.groupby(["LST", "Buffer_m", "Block_size_m"], as_index=False).agg(
        Fold_R2_mean=("test_R2", "mean"), Fold_R2_SD=("test_R2", "std"),
        Fold_RMSE_mean=("test_RMSE", "mean"), Fold_RMSE_SD=("test_RMSE", "std"),
    )
    comparison = comparison.merge(dispersion, on=["LST", "Buffer_m", "Block_size_m"],
                                  how="left", validate="one_to_one")
    comparison = comparison.sort_values(["LST", "Block_size_m"]).reset_index(drop=True)
    save_csv(comparison, root / "spatial_block_size_comparison.csv")
    save_csv(all_folds, root / "all_outer_fold_metrics_and_parameters.csv")
    if cfg_base.export_excel:
        try:
            with pd.ExcelWriter(root / "spatial_block_size_comparison.xlsx", engine="openpyxl") as writer:
                comparison.to_excel(writer, sheet_name="Pooled_OOF_comparison", index=False)
                all_folds.to_excel(writer, sheet_name="Outer_fold_results", index=False)
                pd.concat(plans, ignore_index=True).to_excel(writer, sheet_name="Spatial_fold_plan", index=False)
        except (PermissionError, ImportError) as exc:
            print(f"Excel导出失败，完整CSV与断点均已保存：{exc}；解除占用/权限限制后重跑可从断点导出。", flush=True)
    (root / "README.txt").write_text(
        "500/1000/1500 m空间块大小敏感性分析。\n"
        "模型：lst0603-90m、lst0619-90m、D_LST-300m。样本与特征一致。\n"
        f"各块大小均执行外层{cfg_base.outer_folds}折/内层{cfg_base.inner_folds}折空间调参，{cfg_base.n_iter_search}组随机候选参数，RMSE为选择指标。\n"
        "主要精度为全部外层OOF预测合并计算的R²和RMSE；不是五折R²的均值。\n"
        "折间标准差不是置信区间；不同块大小的同编号折不是配对样本。\n"
        "未设置额外空间排除带，块大小不代表最小训练-测试距离。\n"
        "网格原点固定，未检验网格平移；结果不支持所有空间划分均稳健的断言。\n"
        "1000 m结果在本次数据上重新计算，不混用旧数据对应的历史精度。\n"
        "不自动选择R²最高的块大小，不据此声称已消除空间依赖。\n",
        encoding="utf-8"
    )
    print(comparison[["LST", "Buffer_m", "Block_size_m", "n_samples", "n_spatial_blocks",
                      "Spatial_CV_R2", "Spatial_CV_RMSE", "R2_difference_vs_1000m",
                      "RMSE_difference_vs_1000m"]].to_string(index=False), flush=True)
    print(f"最终汇总：{root}", flush=True)
    return comparison


if __name__ == "__main__":
    base = replace(Config(), n_iter_search=N_ITER_SEARCH, search_jobs=SEARCH_JOBS,
                   run_final_tuning=RUN_FINAL_TUNING)
    main(base_config=base)



ModuleNotFoundError: No module named 'joblib'

In [5]:
import sys
import importlib.metadata as md

print("当前Python：", sys.executable)

for package in [
    "numpy", "pandas", "scipy",
    "scikit-learn", "joblib", "contourpy"
]:
    try:
        dist = md.distribution(package)
        print(
            f"{package}: {dist.version}\n"
            f"位置：{dist.locate_file('')}\n"
        )
    except md.PackageNotFoundError:
        print(f"{package}: 未安装")

当前Python： C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe
numpy: 1.24.3
位置：C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages

pandas: 2.0.2
位置：C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages

scipy: 1.9.3
位置：C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages

scikit-learn: 未安装
joblib: 未安装
contourpy: 1.2.0
位置：C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages



In [3]:
%run "E:/Codex/C-Drive-Workspaces/2026-08-01/jin-ji/RF_three_models_block_size_sensitivity.py"

先预检所有空间块尺度；此阶段不训练随机森林。
  500m：949个共同样本，679个空间块，外层5折，内层5折。
  1000m：949个共同样本，370个空间块，外层5折，内层5折。
  1500m：949个共同样本，225个空间块，外层5折，内层5折。
完整计算：约13,500次CV拟合，另加54次搜索refit。
主输出目录：E:\博士申请\第二篇论文\脚本\Python\importance\three_models_spatial_cv_block_sensitivity_500_1000_1500

空间块尺度 1/3：500 m
输入949个社区；面积>10000：949个；共同完整样本：949个。
679个空间块，3个指定模型组合；首次完整运行预计4,500次CV拟合，另加每次搜索的refit。
指定模型：lst0603–90 m；lst0619–90 m；D_LST–300 m
输出目录：E:\博士申请\第二篇论文\脚本\Python\importance\three_models_spatial_cv_block_sensitivity_500_1000_1500\block_0500m

组合1/3：lst0603, 90m
[lst0603, 90m] 外层折1：读取已完成断点
[lst0603, 90m] 外层折2：读取已完成断点
[lst0603, 90m] 外层折3：读取已完成断点
[lst0603, 90m] 外层折4：读取已完成断点
[lst0603, 90m] 外层折5：读取已完成断点
[lst0603, 90m] 最终调优：读取已完成断点
外层OOF R²=0.5511，RMSE=1.0137；本次已用时0.0分钟

组合2/3：lst0619, 90m
[lst0619, 90m] 外层折1：读取已完成断点
[lst0619, 90m] 外层折2：读取已完成断点
[lst0619, 90m] 外层折3：读取已完成断点
[lst0619, 90m] 外层折4：读取已完成断点
[lst0619, 90m] 外层折5：读取已完成断点
[lst0619, 90m] 最终调优：读取已完成断点
外层OOF R²=0.6147，RMSE=1.1075；本次已用时0.0分钟

组合3/3：D_LST, 300m
[D_LST, 300m] 

In [10]:
from pathlib import Path
from collections import deque

# 训练结果目录
result_dir = Path(
    r"E:\博士申请\第二篇论文\脚本\Python\importance"
    r"\three_models_spatial_cv_block_sensitivity_2000_4000_8000"
)

# 训练日志目录
log_dir = Path(
    r"E:\Codex\C-Drive-Workspaces\2026-08-01\jin-ji"
    r"\block_2_4_8km_training_logs"
)

# 1. 查看各模型已完成的步骤
print("块大小 | 模型    | 已完成外层折 | 最终调参")
print("-" * 55)

for block in (2000, 4000, 8000):
    for target, buffer in (
        ("lst0603", 90),
        ("lst0619", 90),
        ("D_LST", 300),
    ):
        folder = (
            result_dir
            / f"block_{block:04d}m"
            / f"buffer_{buffer:03d}m"
            / target
        )

        completed = len(
            list(folder.glob("outer_fold_*/completed.json"))
        )
        final_done = (
            folder / "final_tuning_completed.json"
        ).exists()

        print(
            f"{block:4d} m | {target:7s} | "
            f"{completed}/5           | "
            f"{'已完成' if final_done else '未完成'}"
        )

# 2. 查看最新日志末尾
logs = list(log_dir.glob("*.stdout.log"))

if logs:
    latest = max(logs, key=lambda p: p.stat().st_mtime)

    print("\n最新训练日志：")
    with latest.open(encoding="utf-8", errors="replace") as file:
        print("".join(deque(file, maxlen=20)))

块大小 | 模型    | 已完成外层折 | 最终调参
-------------------------------------------------------
2000 m | lst0603 | 5/5           | 已完成
2000 m | lst0619 | 5/5           | 已完成
2000 m | D_LST   | 5/5           | 已完成
4000 m | lst0603 | 5/5           | 已完成
4000 m | lst0619 | 5/5           | 已完成
4000 m | D_LST   | 5/5           | 已完成
8000 m | lst0603 | 5/5           | 已完成
8000 m | lst0619 | 5/5           | 已完成
8000 m | D_LST   | 5/5           | 已完成

最新训练日志：
    LST  Block_size_m  Delta_R2_vs_reference  Delta_RMSE_vs_reference  RMSE_change_pct_vs_reference
  D_LST         500.0               0.039641                -0.017967                     -2.949836
  D_LST        1000.0               0.000000                 0.000000                      0.000000
  D_LST        1500.0               0.027938                -0.012607                     -2.069762
  D_LST        2000.0              -0.001273                 0.000568                      0.093289
  D_LST        4000.0              -0.012989            